# engine

> In-process wav2vec 2.0 on Modular MAX: compile cache, embed, trim.


In [ ]:
#| default_exp engine


In [ ]:
#| hide
from nbdev.showdoc import *


An embedding is the mean final hidden state over the clip, minus a fixed "typical audio" vector (`embed_center.npy`), scaled to unit length. Clips are padded up to a whole second because each length is its own compiled graph (a cold compile takes about a minute). Hidden states from the pad are dropped before pooling. MAX is only touched from one worker thread, so loading and every embed run there.


In [ ]:
#| export
import asyncio, base64, os, sys, threading, time, traceback
import numpy as np
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path


In [ ]:
#| export
os.environ.setdefault("MODULAR_HOME", f"{sys.prefix}/share/max")
from nadakit.core import embed_center_path, models_dir

SAMPLE_RATE = sr = 16_000
MAX_SECONDS = maxsec = 10
WARM_SECONDS = 2  # compiled at startup; a typical length, so a cold cache costs the splash and not the first sound
EMBEDDING_SIZE = dim = 768
KEEP_GRAPHS = 4  # as in nadayantra; evicting a length forces a cold compile


In [ ]:
#| export
def pcm(b64):
    "16-bit little-endian PCM from the browser, as float32 in -1..1"
    raw = base64.b64decode(b64)
    raw = raw[: len(raw) // 2 * 2]
    return np.frombuffer(raw, "<i2").astype("f4")[: sr * maxsec] / 32767 if raw else None


In [ ]:
#| export
def trim(x):
    "Cut the quiet lead-in and tail so a sound embeds the same however long you waited"
    if x is None or not x.size or (peak := np.abs(x).max()) < 0.01:
        return None
    loud = np.flatnonzero(np.abs(x) >= max(peak * 0.15, 0.01))
    pad, n = int(sr * 0.1), x.size - 1
    a, b = max(loud[0] - pad, 0), min(loud[-1] + pad, n)
    short = int(sr * 0.5) - (b - a + 1)  # a click gets a little more room
    if short > 0:
        a, b = max(a - short // 2, 0), min(b + short - short // 2, n)
    return x[a : b + 1]


In [ ]:
#| export
class Engine:
    def __init__(self, models: Path | None = None):
        from nadakit.config import base_960h_config
        from nadakit.model import build, load_checkpoint

        models = Path(models) if models else models_dir()
        checkpoint = models / "wav2vec2.safetensors"
        if not checkpoint.is_file():
            raise FileNotFoundError(f"missing model file: {checkpoint}. Run: pixi run convert")

        self.cfg = self.config = base_960h_config()
        self.device, self.graphs = "cpu", OrderedDict()

        center = models / "embed_center.npy"
        if not center.is_file():
            center = embed_center_path()
        if center.is_file():
            self.center = np.load(center).astype(np.float32).reshape(-1)
        else:
            print(f"[nada] warning: {center} is missing; embeddings are not centered", flush=True)
            self.center = np.zeros(EMBEDDING_SIZE, dtype=np.float32)

        t = time.time()
        self.model = build(self.cfg)
        load_checkpoint(self.model, checkpoint)
        print(f"[nada] loaded weights in {time.time() - t:.1f}s", flush=True)
        self.graph(WARM_SECONDS * sr)  # a typical length, so a cold cache costs the splash and not the first sound

    def graph(self, n):
        from nadakit.model import compile_model

        g = self.graphs.pop(n, None)
        if g is None:
            t = time.time()
            g = compile_model(self.model, batch=1, num_samples=n)
            print(f"[nada] compiled {n // sr}s graph in {time.time() - t:.1f}s")
        self.graphs[n] = g
        while len(self.graphs) > KEEP_GRAPHS:
            self.graphs.popitem(last=False)
        return g

    def embed(self, x):
        "Mean-pool real frames, subtract the center, L2-normalize. Same as nadayantra."
        from nadakit.features import normalize
        from nadakit.model import to_device

        x = x[: sr * maxsec]
        if x.size < sr // 10:
            raise ValueError("That sound was too short. Hold it a little longer.")
        real = self.cfg.output_length(x.size)
        x = np.pad(normalize(x), (0, -x.size % sr))
        _, h = self.graph(x.size)(to_device(self.model, x[None]))
        v = np.asarray(h.to_numpy(), dtype="f4")[0, :real].mean(0) - self.center
        return v / max(float(np.linalg.norm(v)), 1e-12)

    embed_audio = embed


In [ ]:
#| export
class Nada:
    "One shared model with a state the UI can poll: idle, starting, ready or failed"

    def __init__(self):
        self.state, self.device, self.error, self.eng = "idle", None, None, None
        self.lock, self.pool = threading.Lock(), ThreadPoolExecutor(1)

    def warm_up(self):
        with self.lock:
            if self.state in ("starting", "ready"):
                return
            self.state, self.error = "starting", None
        self.pool.submit(self._load)

    def _load(self):
        try:
            self.eng = Engine()
            self.device, self.state = self.eng.device, "ready"
        except Exception as e:
            traceback.print_exc()
            self.error, self.state = f"{type(e).__name__}: {e}", "failed"

    def _embed(self, x):
        if not self.eng:
            raise RuntimeError(f"nada could not start. {self.error or ''}")
        return self.eng.embed(x)

    async def embed(self, x):
        if self.state == "idle":
            self.warm_up()
        return await asyncio.get_running_loop().run_in_executor(self.pool, self._embed, x)  # queues behind loading


In [ ]:
#| export
nada = Nada()


In [ ]:
#| hide
n = 12_001
assert n + (-n % sr) == 2 * sr  # pad up to the next whole second, then pool only the real frames